# Задание 4. Fine-tuning.
В качестве модели для дообучения была выбрана модель [Qwen/Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B), а в качестве задачи - определение эмоциональной окраски текста на примере датасета [dair-ai/emotion](https://huggingface.co/datasets/dair-ai/emotion).
## Подготовка данных
Для начала загрузим датасет:

In [2]:
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")
dataset['train'].features

README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'text': Value('string'),
 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}

В датасете представлены тексты с метками, принадлежащими одному из 6 классов: `sadness`, `joy`, `love`, `anger`, `fear`, `surprise`:

In [3]:
dataset['train'][111]

{'text': 'i am feeling all useful', 'label': 1}

Однако количество представителей разных классов в датасете несбалансировано. Больше половины датасета это тексты с метками `sadness` и `joy`, а класс `surprise` занимает всего 3.6% от train.

In [4]:
from collections import Counter
names = dataset['train'].features['label'].names
counts = Counter(dataset['train']['label'])

print(f"{'Label':<9} {'Count':<6} {'Percentage'}")
for idx, count in counts.most_common():
    print(f"{names[idx]:<9} {count:<6} {(count / len(dataset['train']) * 100):.1f}%")

Label     Count  Percentage
joy       5362   33.5%
sadness   4666   29.2%
anger     2159   13.5%
fear      1937   12.1%
love      1304   8.2%
surprise  572    3.6%


Для более стабильного обучения создадим сбалансированную обучающую выборку:

In [5]:
import numpy as np

seed = 67
rng = np.random.default_rng(seed)
label_names = dataset["train"].features["label"].names

train_full = dataset["train"].select(
    sorted(
        i
        for label_id in range(len(label_names))
        for i in rng.choice(
            np.where(np.array(dataset["train"]["label"]) == label_id)[0], 200, replace=False
        )
    )
).shuffle(seed=seed)

test = dataset["test"].select(range(100))
split = train_full.train_test_split(test_size=0.1, seed=seed, stratify_by_column="label")
train, val = split["train"], split["test"]

for name, ds in zip(
    ["train_full", "train", "val", "test"],
    [train_full, train, val, test],
):
    counts = Counter(ds["label"])
    print(name, len(ds))
    print({label_names[i]: counts.get(i, 0) for i in range(len(label_names))})

train_full 1200
{'sadness': 200, 'joy': 200, 'love': 200, 'anger': 200, 'fear': 200, 'surprise': 200}
train 1080
{'sadness': 180, 'joy': 180, 'love': 180, 'anger': 180, 'fear': 180, 'surprise': 180}
val 120
{'sadness': 20, 'joy': 20, 'love': 20, 'anger': 20, 'fear': 20, 'surprise': 20}
test 100
{'sadness': 32, 'joy': 31, 'love': 5, 'anger': 17, 'fear': 12, 'surprise': 3}


## Обучение и оценка моделей

Импортируем необходимые библиотеки и зафиксируем seed для детерминизма:

In [6]:
import gc
import os
import random
import re

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from tempfile import mkdtemp

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

ChatDataset для предсказаний ответов ассистента:

In [7]:
import torch
from torch.utils.data import Dataset

class ChatDataset(Dataset):
    def __init__(self, ds, tokenizer, max_length=256):
        self.data = []

        for row in ds:
            messages = [
                {
                    "role": "system",
                    "content": "You are an emotion classification assistant. Answer with exactly one label: sadness, joy, love, anger, fear, surprise."
                },
                {"role": "user", "content": row["text"]},
                {"role": "assistant", "content": label_names[row["label"]]},
            ]

            full_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            prompt_text = tokenizer.apply_chat_template(
                messages[:-1],
                tokenize=False,
                add_generation_prompt=True,
            )

            ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
            prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

            ids = ids[:max_length]
            labels = ids.copy()

            prompt_len = min(len(prompt_ids), len(labels))
            labels[:prompt_len] = [-100] * prompt_len

            self.data.append({
                "input_ids": ids,
                "attention_mask": [1] * len(ids),
                "labels": labels,
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]

Загрузка токенизатора:

In [8]:
instruct_model_name = "Qwen/Qwen3.5-2B"
base_model_name = "Qwen/Qwen3.5-2B-Base"

reference_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)
if reference_tokenizer.pad_token is None:
    reference_tokenizer.pad_token = reference_tokenizer.eos_token

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Функция оценки модели. В качестве метрики взята макроусредненная F1-мера.

In [9]:
def evaluate_model(model, tokenizer, test_ds):
    model.eval()
    preds = []
    true = test_ds["label"]

    for text in test_ds["text"]:
        messages = [
            {
                "role": "system",
                "content": "You are an emotion classification assistant. Answer with exactly one label: sadness, joy, love, anger, fear, surprise."
            },
            {"role": "user", "content": text},
        ]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        x = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.no_grad():
            y = model.generate(**x, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.pad_token_id,)

        s = tokenizer.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
        s = re.split(r"[\s,.;:!?()\[\]{}]+", s)[0]
        preds.append(label_names.index(s) if s in label_names else -1)

    valid_true = [t for t, p in zip(true, preds) if p != -1]
    valid_pred = [p for p in preds if p != -1]

    metrics = {
        "f1_macro": f1_score(valid_true, valid_pred, average="macro") if valid_pred else 0.0,
    }

    return metrics, preds

Функция дообучения модели. Для дообучения будем использовать LoRA, так как иначе не помещается в 2 T4 видеокарты от Kaggle. Добавим адаптеры ко всевозможным слоям.

In [ ]:
def train_and_eval(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")
    model.config.use_cache = False

    model = get_peft_model(
        model,
        LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        ),
    )

    trainer = Trainer(
        model=model,
        processing_class=tokenizer,
        args=TrainingArguments(
            output_dir=mkdtemp(),
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            gradient_accumulation_steps=2,
            num_train_epochs=3,
            learning_rate=2e-4,
            logging_steps=10,
            eval_strategy="epoch",
            save_strategy="no",
            report_to="none",
            fp16=torch.cuda.is_available(),
        ),
        train_dataset=ChatDataset(train, tokenizer),
        eval_dataset=ChatDataset(val, tokenizer),
        data_collator=DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model,
            pad_to_multiple_of=8,
            label_pad_token_id=-100,
        ),
    )

    trainer.train()

    return evaluate_model(model, tokenizer, test)

Обучение инструктивной модели:

In [10]:
instruct_metrics, instruct_predictions = train_and_eval(instruct_model_name)
pd.DataFrame([{"model": instruct_model_name, **instruct_metrics}])

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Epoch,Training Loss,Validation Loss
1,0.234109,0.203221
2,0.142615,0.136294
3,0.066969,0.127410


,model,f1_macro
0,Qwen/Qwen3.5-2B,0.736406


Обучение базовой модели:

In [11]:
base_metrics, base_predictions = train_and_eval(base_model_name)
pd.DataFrame([{"model": base_model_name, **base_metrics}])

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Epoch,Training Loss,Validation Loss
1,0.242727,0.195782
2,0.136655,0.135132
3,0.067344,0.134231


,model,f1_macro
0,Qwen/Qwen3.5-2B-Base,0.717064


Посчитаем метрики инструктивной модели без дообучения:

In [12]:
tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(instruct_model_name, torch_dtype="auto")
model.config.use_cache = False

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

instruct_raw_metrics, instruct_raw_predictions = evaluate_model(model, tokenizer, test)
pd.DataFrame([{"model": f"{instruct_model_name} (no fine-tuning)", **instruct_raw_metrics}])

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

,model,f1_macro
0,Qwen/Qwen3.5-2B (no fine-tuning),0.480376


## Выводы
Удалось получить следующие результаты:

In [13]:
pd.DataFrame([
    {"model": f"{instruct_model_name} (fine-tuned)", **instruct_metrics},
    {"model": f"{base_model_name} (fine-tuned)", **base_metrics},
    {"model": f"{instruct_model_name} (no fine-tuning)", **instruct_raw_metrics},
]).sort_values("f1_macro", ascending=False).reset_index(drop=True)

,model,f1_macro
0,Qwen/Qwen3.5-2B (fine-tuned),0.736406
1,Qwen/Qwen3.5-2B-Base (fine-tuned),0.717064
2,Qwen/Qwen3.5-2B (no fine-tuning),0.480376


Видно, что после дообучения модели стали работать лучше. Разница в F1-мере больше 0.2 у обеих моделей (по сравнению с instruct-моделью без дообучения). При этом дообученная изначально инструктивная модель показала лучший результат. Возможно, это связано с тем, что инструктивная модель уже обучена следованию инструкциям, в то время как базовая модель обучалась только предсказанию последующих токенов, и для нее, вероятно, нужно больше данных для обучения именно ответам в формате чата.